# Week 6 · Day 3 — Hypothesis tests I: is the difference real?

The two-proportion z-test written by hand, then the library versions; chi-square, Welch, paired; effect sizes; the risk-committee note.

In [1]:
from pathlib import Path
import hashlib
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")           # this notebook lives in data/python/week6/, so "../.." is the data/ folder
AS_OF = pd.Timestamp("2026-10-01")
con = sqlite3.connect(DATA / "raw" / "nur.db")
rng = np.random.default_rng(2026)   # one seeded generator: the same "random" numbers on every run


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w6:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

In [2]:
# The three frames of the week (built exactly as in week 5; run, do not edit)
loans = pd.read_sql("""
    SELECT l.loan_id, l.disbursed_at, l.principal, l.term_months, l.risk_grade, l.status,
           p.code, p.name AS product, a.channel, a.credit_score, b.branch_name
    FROM loans l JOIN products p USING (product_id) JOIN applications a USING (application_id)
    JOIN branches b ON b.branch_id = l.branch_id""", con, parse_dates=["disbursed_at"])
inst = pd.read_sql("SELECT * FROM installments", con, parse_dates=["due_date"])
pay = (pd.read_sql("SELECT * FROM payments", con, parse_dates=["paid_at"])
         .sort_values("payment_id").drop_duplicates(["installment_id", "paid_at", "amount"]))   # 52 850 clean payments

# f: one row per loan whose FIRST instalment fell due on or before as-of - 30 days; fpd30 = 1 if not fully paid within 30 days
first = inst[(inst["installment_no"] == 1) & (inst["due_date"] <= AS_OF - pd.Timedelta(days=30))]
paid1 = pay[pay["paid_at"] <= AS_OF].groupby("installment_id", as_index=False).agg(paid=("amount", "sum"), first_paid=("paid_at", "min"))
f = first.merge(paid1, on="installment_id", how="left").fillna({"paid": 0})
f["fpd30"] = ((f["paid"] < f["total_due"]) | ((f["first_paid"] - f["due_date"]).dt.days > 30)).astype(int)
f = f.merge(loans, on="loan_id", validate="1:1")

# x: one row per clean payment, with its due date, days late and the loan's grade / product
x = pay.merge(inst[["installment_id", "due_date"]], on="installment_id").merge(
        loans[["loan_id", "risk_grade", "code", "channel"]], on="loan_id")
x["days_late"] = (x["paid_at"].dt.normalize() - x["due_date"]).dt.days
x["late"] = (x["paid_at"] > x["due_date"]).astype(int)
assert len(loans) == 7828 and len(f) == 7099 and f["fpd30"].sum() == 91 and len(x) == 52850
print(f"loans {len(loans)} · FPD30 {f.fpd30.sum()} / {len(f)} = {f.fpd30.mean():.2%} · clean payments {len(x)}")

loans 7828 · FPD30 91 / 7099 = 1.28% · clean payments 52850


## 1. `two_prop_ztest(k1, n1, k2, n2)` — pooled z and the two-sided p-value

In [3]:
def two_prop_ztest(k1, n1, k2, n2):
    """(z, p_two_sided) for H0: p1 = p2, using the pooled proportion in the standard error."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [4]:
z, p = two_prop_ztest(41, 2411, 34, 3263)
check(z, ['16231f830003', '1da909bb68da', '3376deb071c9', '511da3093bf5', '6384fcb4ec6d', '6909444f3244', '72b35dfdfe46', '72bb5d26c40e', '75b524fb6d46', '7766b14fb657', '7b280521f94d', '8a6847f7d140', 'a250f8212e56', 'bc619616ffe3', 'c35ee76654ff', 'd6e1632226fa', 'e15e8c679266', 'e658c4e40d55', 'eeac45ff5ad2', 'f83cec266418', 'fea2b2f9cf13'], 4); check(p, ['0884282fe700', '0fd0b0e2c62e', '1c981dabf0d0', '21e0be13c0e2', '2599a9766330', '260124c7d81f', '31194297e3cc', '319b88ff039a', '390eba67ddff', '40aace45de42', '4892c1cffdac', '48a939952f07', '4de4613e5d68', '5a8bcd450f8e', '717a6d2dc41d', '7b9d35167eab', '7ef896b566ff', 'a5d30e8c954f', 'ba1022793a35', 'd6e8786f455f', 'e14ac4981acd'], 4)
print(f"two_prop_ztest OK: z = {z:.3f}, p = {p:.4f}")

NotImplementedError: TODO: write me, then run the assert cell below

## 1b. The library versions agree

In [5]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
print(proportions_ztest([41, 34], [2411, 3263]))
print(proportions_ztest([41, 34], [2411, 3263], alternative="larger"))
print("95 % CI of the difference (Newcombe):", confint_proportions_2indep(41, 2411, 34, 3263, method="newcomb"))

(np.float64(2.14712272235006), np.float64(0.03178351278354667))
(np.float64(2.14712272235006), np.float64(0.015891756391773336))
95 % CI of the difference (Newcombe): (np.float64(0.000534731542129927), np.float64(0.013256972501504554))


## 2. Chi-square: FPD × channel (three channels at once) and late payment × grade

In [6]:
ct = pd.crosstab(f["channel"], f["fpd30"])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(ct, f"\nchi2 = {chi2:.3f}, dof = {dof}, p = {p:.4f}")
print("expected counts under H0:\n", expected.round(1))

fpd30       0   1
channel          
app      2370  41
branch   3229  34
partner  1409  16 
chi2 = 5.108, dof = 2, p = 0.0778
expected counts under H0:
 [[2380.1   30.9]
 [3221.2   41.8]
 [1406.7   18.3]]


In [7]:
ct2 = pd.crosstab(x["risk_grade"], x["late"])
chi2, p, dof, _ = stats.chi2_contingency(ct2)
print(ct2.assign(late_share=(ct2[1] / ct2.sum(axis=1)).round(4)), f"\nchi2 = {chi2:.1f}, dof = {dof}, p = {p:.3g}")
print("Cramér's V:", round(np.sqrt(chi2 / (ct2.to_numpy().sum() * (min(ct2.shape) - 1))), 3))

late            0     1  late_share
risk_grade                         
A           11597   622      0.0509
B           21248  2313      0.0982
C           11798  2879      0.1962
D            1541   852      0.3560 
chi2 = 2595.0, dof = 3, p = 0
Cramér's V: 0.222


In [8]:
# Applicants' credit scores by channel — used in sections 3 and 4
apps = pd.read_sql("SELECT channel, credit_score FROM applications", con)
sa = apps.loc[apps.channel == "app", "credit_score"].dropna()
sb = apps.loc[apps.channel == "branch", "credit_score"].dropna()
print(f"app {len(sa)} scores, mean {sa.mean():.1f} · branch {len(sb)} scores, mean {sb.mean():.1f}")

app 4874 scores, mean 661.0 · branch 6049 scores, mean 659.6


## 3. Effect sizes — `relative_risk(k1, n1, k2, n2)` and `cohens_d(a, b)`

In [9]:
def relative_risk(k1, n1, k2, n2):
    """(k1/n1) / (k2/n2)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [10]:
def cohens_d(a, b):
    """(mean(a) - mean(b)) / pooled sd, pooled sd = sqrt(((na-1)*var_a + (nb-1)*var_b) / (na+nb-2)), ddof=1."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [11]:
check(relative_risk(41, 2411, 34, 3263), ['77bdad840db6', 'ec2784a03d81', 'ecb398f5da79'], 3)
check(cohens_d(sa, sb), ['01d063cca092', '051bc84c6beb', '060a93689058', '091cea7ada83', '1f3d7e81bae2', '3a4bd4e06353', '3de1fbae5ed9', '4e593ec09857', '68dc409d8af3', '7cc66499c3ce', '9cc93ff9f59a', 'aa720a456482', 'b4c2b7e108b7', 'da6e5bcbe008', 'ddf82c919d0e', 'e033c4f32de4', 'e062f9373c1f', 'e40a726321b1', 'e42f25bcc11f', 'ec63562e6b4c', 'ff3f0068e49b'], 4)
print("effect sizes OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 4. Welch t-test, Levene, Shapiro — applicants' credit score, app vs branch; then the approved loans' scores

In [12]:
print("Welch:", stats.ttest_ind(sa, sb, equal_var=False))
print("Levene:", stats.levene(sa, sb))
print("Shapiro on 500 app applicants:", stats.shapiro(sa.sample(500, random_state=2026)))
la = loans.loc[loans.channel == "app", "credit_score"].dropna()
lb = loans.loc[loans.channel == "branch", "credit_score"].dropna()
print(f"loans: app {la.mean():.1f} vs branch {lb.mean():.1f}", stats.ttest_ind(la, lb, equal_var=False))

Welch: TtestResult(statistic=np.float64(1.2327645308939308), pvalue=np.float64(0.21769135726915906), df=np.float64(10451.089015792712))
Levene: LeveneResult(statistic=np.float64(0.08903856671518454), pvalue=np.float64(0.7654086723104475))
Shapiro on 500 app applicants: ShapiroResult(statistic=np.float64(0.9976072207927804), pvalue=np.float64(0.6990563994307386))
loans: app 683.2 vs branch 679.8 TtestResult(statistic=np.float64(2.4817485501572887), pvalue=np.float64(0.013102693814806507), df=np.float64(5671.492393745764))


## 5. Paired: the same ten branches' FPD30, 2025 vs 2026 vintages

In [13]:
f["year"] = f["disbursed_at"].dt.year
w = f[f.year.isin([2025, 2026])].pivot_table(index="branch_name", columns="year", values="fpd30", aggfunc="mean") * 100
print(w.round(2))
print("paired:     ", stats.ttest_rel(w[2026], w[2025]))
print("independent:", stats.ttest_ind(w[2026], w[2025]))

year               2025  2026
branch_name                  
Andijan Central    1.57  0.89
Bukhara Central    3.20  1.99
Fergana Central    1.22  0.82
Namangan Central   1.95  0.49
Nukus Central      1.12  0.66
Samarkand Central  0.87  0.75
Tashkent Central   1.94  1.44
Tashkent North     0.55  1.54
Tashkent South     1.60  1.03
Termez Central     2.09  0.72
paired:      TtestResult(statistic=np.float64(-2.561228391446596), pvalue=np.float64(0.030625346197491975), df=np.int64(9))
independent: TtestResult(statistic=np.float64(-2.0530368028268424), pvalue=np.float64(0.05489613270595176), df=np.float64(18.0))


## 6. The risk-committee note (Markdown, six lines)
Question · Metric definition · Numbers with 95 % CIs · Test + p-value · Effect in business terms (extra FPDs per 1 000 loans) · Decision.

…